# WCL Bronze → Silver Pipeline
**Metadata-driven | SCD1/SCD2 | FULL/INCREMENTAL | NATURAL/HASH PK | Config-controlled transformations**

| Cell | Purpose |
|------|---------|
| 1 | Widgets — runtime parameters |
| 2 | Imports |
| 3 | Constants & config from widgets |
| 4 | `write_audit_log` |
| 5 | `write_error_log` + `log_dq_failures` |
| 6 | `get_last_watermark` + `update_watermark` |
| 7 | `load_table_configs` |
| 8 | `load_column_mapping` |
| 9 | `load_dq_rules` |
| 10 | `apply_column_mapping` — metadata-driven transformations |
| 11 | `run_dq_checks` |
| 12 | `deduplicate` |
| 13 | `generate_hash_key` |
| 14 | `apply_active_filter` |
| 15 | `apply_incremental_filter` |
| 16 | `merge_scd1` |
| 17 | `merge_scd2` |
| 18 | `process_table` |
| 19 | `main` |
| 20 | Run |


In [0]:
# ── Widgets — Runtime Parameters ─────────────────────────────────────────────
# These allow overriding defaults from Databricks job parameters or UI

dbutils.widgets.removeAll()

dbutils.widgets.text(
    name    = "catalog",
    defaultValue = "Catalog_Name",
    label   = "Catalog"
)
dbutils.widgets.text(
    name    = "source_schema",
    defaultValue = "bronze",
    label   = "Source Schema"
)
dbutils.widgets.text(
    name    = "target_schema",
    defaultValue = "silver",
    label   = "Target Schema"
)
dbutils.widgets.text(
    name    = "meta_schema",
    defaultValue = "meta",
    label   = "Meta Schema"
)

dbutils.widgets.dropdown(
    name    = "run_mode",
    defaultValue = "normal",
    choices = ["normal", "force_full"],
    label   = "Run Mode (force_full overrides all tables to FULL load)"
)
dbutils.widgets.text(
    name    = "table_filter",
    defaultValue = "",
    label   = "Table Filter (comma-separated source table names, empty = all)"
)

print("Widgets ready.")


In [0]:
# ── Imports ───────────────────────────────────────────────────────────────────
import uuid
import json
import traceback
from datetime import datetime, timezone

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, BooleanType,
    TimestampType, DateType, DecimalType, FloatType,
    DoubleType
)
from delta.tables import DeltaTable

print("Imports complete.")


In [0]:
# ── Constants — sourced from widgets ─────────────────────────────────────────
CATALOG        = dbutils.widgets.get("catalog")
SOURCE_SCHEMA  = dbutils.widgets.get("source_schema")
TARGET_SCHEMA  = dbutils.widgets.get("target_schema")
META_SCHEMA    = dbutils.widgets.get("meta_schema")
PIPELINE_ID    = 1 # Bronze → Silver pipeline identifier
RUN_MODE       = dbutils.widgets.get("run_mode")          # normal | force_full
TABLE_FILTER   = dbutils.widgets.get("table_filter")       # empty = process all

# One UUID per full notebook run — shared across all tables
BATCH_ID       = str(uuid.uuid4())
RUN_TIMESTAMP  = datetime.now(timezone.utc)

# SCD2 audit columns — excluded from change-detection comparison
SCD2_COLS      = ["effective_from", "effective_to", "is_current",
                   "dwh_created_at", "dwh_updated_at"]

# Data type cast map
DTYPE_MAP = {
    "STRING"    : StringType(),
    "VARCHAR"   : StringType(),
    "INT"       : IntegerType(),
    "INTEGER"   : IntegerType(),
    "BIGINT"    : LongType(),
    "LONG"      : LongType(),
    "BOOLEAN"   : BooleanType(),
    "BOOL"      : BooleanType(),
    "TIMESTAMP" : TimestampType(),
    "DATE"      : DateType(),
    "FLOAT"     : FloatType(),
    "DOUBLE"    : DoubleType(),
}

# Standardization maps — value corrections per target column
STANDARDIZE_MAP = {
    "order_status"     : {"Reject": "Rejected"},
    "approval_status"  : {"Canceled": "Cancelled", "Reject": "Rejected"},
    "trip_status"      : {"CLOSE": "CLOSED", "Open": "OPEN"},
    "trip_type"        : {"OUTBOUND": "Outbound", "INBOUND": "Inbound"},
    "delivery_mode"    : {"undefined": None},
}

print(f"Batch ID      : {BATCH_ID}")
print(f"Run Timestamp : {RUN_TIMESTAMP}")
print(f"Catalog       : {CATALOG}")
print(f"Source Schema : {SOURCE_SCHEMA}")
print(f"Target Schema : {TARGET_SCHEMA}")
print(f"Meta Schema   : {META_SCHEMA}")
print(f"Pipeline ID   : {PIPELINE_ID}")
print(f"Run Mode      : {RUN_MODE}")
print(f"Table Filter  : {TABLE_FILTER if TABLE_FILTER else '(all tables)'}")


In [0]:
# ── write_audit_log ───────────────────────────────────────────────────────────
AUDIT_SCHEMA = StructType([
    StructField("batch_id",         StringType(),    True),
    StructField("pipeline_id",      IntegerType(),   True),
    StructField("source_table",     StringType(),    True),
    StructField("target_table",     StringType(),    True),
    StructField("source_layer",     StringType(),    True),
    StructField("target_layer",     StringType(),    True),
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("records_read",     LongType(),      True),
    StructField("records_inserted", LongType(),      True),
    StructField("records_updated",  LongType(),      True),
    StructField("records_rejected", LongType(),      True),
    StructField("records_deleted",  LongType(),      True),
    StructField("dq_pass_count",    LongType(),      True),
    StructField("dq_fail_count",    LongType(),      True),
    StructField("load_type",        StringType(),    True),
    StructField("write_mode",       StringType(),    True),
    StructField("scd_type",         StringType(),    True),
    StructField("pk_strategy",      StringType(),    True),
    StructField("was_forced_full",  BooleanType(),   True),
    StructField("status",           StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("created_at",       TimestampType(), True),
])


def write_audit_log(
    batch_id, pipeline_id, source_table, target_table,
    source_layer, target_layer, start_time, end_time,
    records_read, records_inserted, records_updated,
    records_rejected, records_deleted,
    dq_pass_count, dq_fail_count,
    load_type, write_mode, scd_type, pk_strategy,
    was_forced_full, status, error_message=""
):
    """
    Write one audit row per table per run.
    Uses explicit schema to prevent Spark type inference mismatches.
    """
    audit_data = [{
        "batch_id"        : str(batch_id),
        "pipeline_id"     : int(pipeline_id),
        "source_table"    : str(source_table),
        "target_table"    : str(target_table),
        "source_layer"    : str(source_layer),
        "target_layer"    : str(target_layer),
        "start_time"      : start_time,
        "end_time"        : end_time,
        "records_read"    : int(records_read      or 0),
        "records_inserted": int(records_inserted  or 0),
        "records_updated" : int(records_updated   or 0),
        "records_rejected": int(records_rejected  or 0),
        "records_deleted" : int(records_deleted   or 0),
        "dq_pass_count"   : int(dq_pass_count     or 0),
        "dq_fail_count"   : int(dq_fail_count     or 0),
        "load_type"       : str(load_type   or ""),
        "write_mode"      : str(write_mode  or ""),
        "scd_type"        : str(scd_type    or ""),
        "pk_strategy"     : str(pk_strategy or ""),
        "was_forced_full" : bool(was_forced_full),
        "status"          : str(status),
        "error_message"   : str(error_message or "")[:8000],
        "created_at"      : datetime.now(timezone.utc)
    }]

    audit_df = spark.createDataFrame(audit_data, schema=AUDIT_SCHEMA)
    audit_df.write.format("delta").mode("append") \
        .saveAsTable(f"{CATALOG}.{META_SCHEMA}.pipeline_audit_log")

    print(f"  [AUDIT] {source_table} → {status} | "
          f"read={records_read} | inserted={records_inserted} | "
          f"updated={records_updated} | rejected={records_rejected} | "
          f"deleted={records_deleted}")

In [0]:
# ── write_error_log + log_dq_failures ────────────────────────────────────────
def write_error_log(batch_id, table_name, record_id, check_type,
                    check_description, severity, raw_record_json):
    """Write a single error record to error_log_table."""
    error_data = [{
        "batch_id"         : str(batch_id),
        "table_name"       : str(table_name),
        "record_id"        : str(record_id),
        "check_type"       : str(check_type),
        "check_description": str(check_description),
        "severity"         : str(severity),
        "raw_record"       : str(raw_record_json),
        "created_at"       : datetime.now(timezone.utc)
    }]
    error_df = spark.createDataFrame(error_data)
    error_df.write.format("delta").mode("append") \
        .saveAsTable(f"{CATALOG}.{META_SCHEMA}.error_log_table")


def log_dq_failures(df_failed: DataFrame, table_name: str,
                    pk_col: str, check_type: str,
                    check_description: str, severity: str,
                    batch_id: str):
    """
    Bulk write DQ failures from a DataFrame to error_log_table.
    Serialises each failing record to JSON for debugging.
    """
    if df_failed.count() == 0:
        return

    df_logged = (
        df_failed
        .withColumn("batch_id",          F.lit(batch_id))
        .withColumn("table_name",        F.lit(table_name))
        .withColumn("record_id",         F.col(pk_col).cast(StringType()))
        .withColumn("check_type",        F.lit(check_type))
        .withColumn("check_description", F.lit(check_description))
        .withColumn("severity",          F.lit(severity))
        .withColumn("raw_record",        F.to_json(F.struct(*df_failed.columns)))
        .withColumn("created_at",        F.current_timestamp())
        .select("batch_id", "table_name", "record_id",
                "check_type", "check_description",
                "severity", "raw_record", "created_at")
    )

    df_logged.write.format("delta").mode("append") \
        .saveAsTable(f"{CATALOG}.{META_SCHEMA}.error_log_table")


In [0]:
# ── get_last_watermark + update_watermark ────────────────────────────────────
def get_last_watermark(source_table: str):
    """
    Read last_watermark_value from metadata_silver_config.
    Returns MAX(watermark_column) stored from last successful run.
    Returns None on first run — triggers full load.
    """
    try:
        result = spark.sql(f"""
            SELECT last_watermark_value
            FROM {CATALOG}.{META_SCHEMA}.metadata_silver_config
            WHERE source_table = '{source_table}'
            AND   is_active = true
            LIMIT 1
        """).collect()

        if result and result[0]["last_watermark_value"] is not None:
            return result[0]["last_watermark_value"]
        return None
    except Exception as e:
        print(f"  [WATERMARK] Could not read watermark for {source_table}: {e}")
        return None


def update_watermark(source_table: str, max_watermark, end_time):
    """
    After a successful run write back to metadata_silver_config:
      last_watermark_value = MAX(watermark_column) from the loaded data
      last_successful_run  = pipeline end_time
    This is what drives the next incremental filter window.
    """
    try:
        spark.sql(f"""
            UPDATE {CATALOG}.{META_SCHEMA}.metadata_silver_config
            SET last_watermark_value = CAST('{max_watermark}' AS TIMESTAMP),
                last_successful_run  = CAST('{end_time}'      AS TIMESTAMP),
                modified_date        = current_timestamp()
            WHERE source_table = '{source_table}'
            AND   is_active    = true
        """)
        print(f"  [WATERMARK] Saved → {source_table} | max_watermark={max_watermark}")
    except Exception as e:
        print(f"  [WATERMARK] Failed to update watermark for {source_table}: {e}")


In [0]:
# ── load_table_configs ────────────────────────────────────────────────────────
def load_table_configs() -> list:
    """
    Load all active table configs from metadata_silver_config.
    Applies table_filter widget if provided.
    Applies run_mode=force_full by overriding load_type at runtime.
    """
    base_query = f"""
        SELECT *
        FROM {CATALOG}.{META_SCHEMA}.metadata_silver_config
        WHERE is_active = true
        ORDER BY load_sequence
    """
    configs = spark.sql(base_query).collect()

    # Apply table_filter widget — comma-separated source table names
    if TABLE_FILTER.strip():
        filter_list = [t.strip().lower() for t in TABLE_FILTER.split(",")]
        configs = [c for c in configs if c["source_table"].lower() in filter_list]

    # Apply force_full run mode — override load_type for all tables
    if RUN_MODE == "force_full":
        configs = [dict(c.asDict(), load_type="FULL") for c in configs]
        print(f"[CONFIG] RUN_MODE=force_full — all tables overridden to FULL load")

    print(f"[CONFIG] Loaded {len(configs)} active table configuration(s).")
    return configs


In [0]:
# ── load_column_mapping ───────────────────────────────────────────────────────
def load_column_mapping(source_table: str, target_table: str) -> list:
    """
    Load column-level mapping including transformation rules
    from metadata_column_config.
    """
    mappings = spark.sql(f"""
        SELECT source_column, target_column,
            source_data_type, target_data_type,
            transformation,
            column_sequence
        FROM {CATALOG}.{META_SCHEMA}.metadata_column_config
        WHERE source_table = '{source_table}'
        AND   target_table = '{target_table}'
        AND   is_active    = true
        ORDER BY column_sequence
    """).collect()
    return mappings


In [0]:
# ── load_dq_rules ─────────────────────────────────────────────────────────────
def load_dq_rules(target_table: str) -> list:
    """
    Load DQ rules for a silver table from meta_data_quality.
    Rules are filtered by target_table_name and is_active=true.
    """
    rules = spark.sql(f"""
        SELECT dq_id, source_column, target_column,
               rule_type, rule_expression, severity
        FROM {CATALOG}.{META_SCHEMA}.meta_data_quality
        WHERE target_table_name = '{target_table}'
        AND   is_active         = true
        ORDER BY dq_id
    """).collect()
    return rules


In [0]:
def apply_column_mapping(df: DataFrame, mappings: list,
                          passthrough_cols: list = None) -> DataFrame:
    """
    Metadata-driven column transformation engine.

    For each column mapping row:
      1. Rename source → target
      2. Apply transformation from metadata_column_config.transformation:
           Y_N_TO_BOOL  : Y/TRUE/1 → True, N/FALSE/0 → False
           INT_TO_BOOL  : 1 → True, 0 → False
           UPPER        : uppercase
           LOWER        : lowercase
           TRIM         : strip whitespace (applied to all strings by default)
           NULL_IF_EMPTY: empty string → NULL (applied to all strings by default)
           STANDARDIZE  : apply STANDARDIZE_MAP correction for the column
      3. Cast to target_data_type
      4. Handle DECIMAL precision/scale from type string e.g. DECIMAL(18,2)

    Empty strings → NULL is applied to ALL string columns regardless of
    transformation setting.

    passthrough_cols : list of column names to carry through UNCHANGED,
                        in addition to the mapped columns. Used for
                        generated columns (e.g. HASH surrogate key) that
                        exist in df but have no entry in metadata_column_config.
    """
    select_exprs = []

    for m in mappings:
        src            = m["source_column"]
        tgt            = m["target_column"]
        transformation = (m["transformation"] or "").upper().strip()
        raw_dtype      = (m["target_data_type"] or "STRING").upper().strip()
        dtype          = raw_dtype.split("(")[0].strip()

        # ── Start with source column ──────────────────────────
        col_expr = F.col(src)

        # ── Transformation logic (metadata-driven) ─────────────
        if transformation == "Y_N_TO_BOOL":
            col_expr = (
                F.when(F.upper(F.trim(F.col(src))).isin("Y", "TRUE", "1"),  F.lit(True))
                 .when(F.upper(F.trim(F.col(src))).isin("N", "FALSE", "0"), F.lit(False))
                 .otherwise(F.lit(None).cast(BooleanType()))
            )

        elif transformation == "INT_TO_BOOL":
            col_expr = (
                F.when(F.col(src) == 1, F.lit(True))
                 .when(F.col(src) == 0, F.lit(False))
                 .otherwise(F.lit(None).cast(BooleanType()))
            )

        elif transformation == "UPPER":
            col_expr = F.upper(F.trim(F.col(src)))
            col_expr = F.when(col_expr == "", F.lit(None)).otherwise(col_expr)

        elif transformation == "LOWER":
            col_expr = F.lower(F.trim(F.col(src)))
            col_expr = F.when(col_expr == "", F.lit(None)).otherwise(col_expr)

        elif transformation == "STANDARDIZE":
            col_expr = F.trim(F.col(src))
            if tgt in STANDARDIZE_MAP:
                for old_val, new_val in STANDARDIZE_MAP[tgt].items():
                    col_expr = F.when(col_expr == old_val, F.lit(new_val)).otherwise(col_expr)
            col_expr = F.when(col_expr == "", F.lit(None)).otherwise(col_expr)

        elif transformation == "NULL_IF_EMPTY":
            col_expr = F.trim(F.col(src))
            col_expr = F.when(col_expr == "", F.lit(None)).otherwise(col_expr)

        elif transformation == "TRIM" or (dtype in ("STRING", "VARCHAR") and not transformation):
            # Default for all string columns: TRIM + NULL_IF_EMPTY
            col_expr = F.trim(F.col(src))
            col_expr = F.when(col_expr == "", F.lit(None)).otherwise(col_expr)

        # ── Cast to target data type ───────────────────────────
        if transformation not in ("Y_N_TO_BOOL", "INT_TO_BOOL"):
            if dtype == "DECIMAL":
                if "(" in raw_dtype:
                    parts = raw_dtype.replace("DECIMAL(", "").replace(")", "").split(",")
                    p, s  = int(parts[0].strip()), int(parts[1].strip())
                else:
                    p, s  = 18, 2
                col_expr = col_expr.cast(DecimalType(p, s))
            elif dtype in DTYPE_MAP:
                col_expr = col_expr.cast(DTYPE_MAP[dtype])

        select_exprs.append(col_expr.alias(tgt))

    # ── Pass through extra columns unchanged ───────────────────
    # Used for generated columns (e.g. HASH surrogate key) that have
    # no entry in metadata_column_config but must survive the select.
    if passthrough_cols:
        for col_name in passthrough_cols:
            if col_name in df.columns:
                select_exprs.append(F.col(col_name))
            else:
                print(f"  [WARN] passthrough_col '{col_name}' not found in DataFrame — skipping")

    return df.select(select_exprs)

In [0]:
def run_dq_checks(df: DataFrame, rules: list, pk_col: str,
                  table_name: str, batch_id: str):
    """
    Run all DQ rules against the DataFrame.
    Returns:
        df_clean   : records passing all HIGH severity checks
        total_pass : count of rule-record passes
        total_fail : count of rule-record failures

    Fix: NULL PKs cannot be excluded via left_anti join (NULL != NULL in joins).
         They are collected separately and filtered with isNotNull().
    """
    df_critical_pks  = None   # non-NULL PK values to exclude
    has_null_pk_fail = False  # flag if any HIGH rule found NULL PKs
    total_pass       = 0
    total_fail       = 0

    for rule in rules:
        rule_type = rule["rule_type"]
        rule_expr = rule["rule_expression"]
        severity  = rule["severity"]
        tgt_col   = rule["target_column"]
        dq_id     = rule["dq_id"]

        try:
            df_fail    = df.filter(f"NOT ({rule_expr})")
            fail_count = df_fail.count()
            pass_count = df.count() - fail_count

            total_pass += pass_count
            total_fail += fail_count

            if fail_count > 0:
                print(f"    [DQ-{rule_type}] col={tgt_col} | severity={severity} | failed={fail_count}")

                # Log all failures to error_log_table
                log_dq_failures(
                    df_failed         = df_fail,
                    table_name        = table_name,
                    pk_col            = pk_col,
                    check_type        = rule_type,
                    check_description = f"dq_id={dq_id}: {rule_expr}",
                    severity          = severity,
                    batch_id          = batch_id
                )

                # Accumulate HIGH severity rejections
                if severity == "HIGH":
                    # Separate NULL PKs from non-NULL PKs
                    # NULL PKs cannot be excluded via join — handle separately
                    null_pk_rows    = df_fail.filter(F.col(pk_col).isNull())
                    non_null_pk_rows = df_fail.filter(F.col(pk_col).isNotNull())

                    if null_pk_rows.count() > 0:
                        has_null_pk_fail = True

                    if non_null_pk_rows.count() > 0:
                        pks = non_null_pk_rows.select(pk_col)
                        df_critical_pks = (
                            pks if df_critical_pks is None
                            else df_critical_pks.union(pks)
                        )

        except Exception as e:
            print(f"    [DQ-ERROR] Rule dq_id={dq_id} failed to execute: {e}")

    # Build clean DataFrame — remove all HIGH severity failures
    df_clean = df

    # Step 1: Remove non-NULL PK failures via left_anti join
    if df_critical_pks is not None:
        df_clean = df_clean.join(
            df_critical_pks.distinct(),
            on  = pk_col,
            how = "left_anti"
        )

    # Step 2: Remove NULL PK records — left_anti misses these
    # NULL PK records can never be merged (Delta NOT NULL constraint)
    if has_null_pk_fail:
        df_clean = df_clean.filter(F.col(pk_col).isNotNull())

    return df_clean, total_pass, total_fail

In [0]:
# ── deduplicate ───────────────────────────────────────────────────────────────
def deduplicate(df: DataFrame, primary_key: str,
                watermark_col_mapped: str) -> DataFrame:
    """
    Deduplicate keeping the latest record per primary key.
    Handles composite keys (comma-separated).

    Order by:
      - watermark_col_mapped DESC if the column exists in df
      - Otherwise first PK column DESC (arbitrary but consistent)

    NOTE: primary_key and watermark_col_mapped are TARGET (mapped) column names.
    This function always runs after apply_column_mapping().
    """
    pk_cols   = [c.strip() for c in primary_key.split(",")]
    order_col = (
        watermark_col_mapped
        if (watermark_col_mapped and watermark_col_mapped in df.columns)
        else pk_cols[0]
    )

    window   = Window.partitionBy(*pk_cols).orderBy(F.col(order_col).desc())
    df_dedup = (
        df.withColumn("_rn", F.row_number().over(window))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )
    return df_dedup


In [0]:
# ── generate_hash_key ────────────────────────────────────────────────────────
def generate_hash_key(df: DataFrame, hash_columns: str,
                      hash_column_name: str) -> DataFrame:
    """
    Generate a SHA-256 surrogate key from multiple SOURCE columns.
    Used when pk_strategy=HASH and no reliable natural PK exists.

    hash_columns     : comma-separated SOURCE column names (before mapping)
    hash_column_name : name of the generated column e.g. surrogate_key

    The hash is computed BEFORE column mapping so it uses original source names.
    NULL values are coalesced to empty string before hashing to avoid
    NULL propagation making the entire hash NULL.

    Result is a 64-character hex string (SHA-256).
    """
    cols = [c.strip() for c in hash_columns.split(",")]

    # Coalesce NULLs to empty string, cast to string, concatenate with separator
    hash_input = F.concat_ws(
        "||",
        *[F.coalesce(F.col(c).cast(StringType()), F.lit("")) for c in cols]
    )
    return df.withColumn(hash_column_name, F.sha2(hash_input, 256))


In [0]:
# ── apply_active_filter ──────────────────────────────────────────────────────
def apply_active_filter(df: DataFrame,
                         active_filter_column: str,
                         active_filter_value: str) -> DataFrame:
    """
    Filter out soft-deleted records before processing.
    Configured via metadata_silver_config columns:
      active_filter_column : e.g. IsDelete, is_active
      active_filter_value  : e.g. false, 0, N, true, 1, Y

    Only records where active_filter_column == active_filter_value are kept.

    Example:
      CRM table with IsDelete column:
        active_filter_column = IsDelete
        active_filter_value  = false
      → keeps only rows where IsDelete = false (i.e. not deleted)
    """
    if not active_filter_column or not active_filter_value:
        return df

    if active_filter_column not in df.columns:
        print(f"  [FILTER] active_filter_column '{active_filter_column}' "
              f"not found in DataFrame — skipping active filter")
        return df

    before_count = df.count()
    df_filtered  = df.filter(
        F.col(active_filter_column).cast(StringType()) == str(active_filter_value)
    )
    after_count  = df_filtered.count()

    if before_count != after_count:
        print(f"  [ACTIVE FILTER] {active_filter_column}={active_filter_value} "
              f"| removed {before_count - after_count} soft-deleted records")

    return df_filtered


In [0]:
# ── apply_incremental_filter ─────────────────────────────────────────────────
def apply_incremental_filter(df: DataFrame,
                              watermark_col: str,
                              last_watermark) -> DataFrame:
    """
    Filter bronze records where watermark_col > last_watermark_value.

    IMPORTANT: watermark_col is the SOURCE column name (before mapping)
    because this filter is applied on the bronze DataFrame BEFORE
    apply_column_mapping() runs.

    Falls back to full load (returns df unchanged) when:
      - last_watermark is None (first run — no prior watermark stored)
      - watermark_col does not exist in the DataFrame
    """
    if last_watermark is None:
        print(f"  [WATERMARK] No prior watermark — performing FULL LOAD")
        return df

    if watermark_col not in df.columns:
        print(f"  [WATERMARK] Column '{watermark_col}' not found — performing FULL LOAD")
        return df

    print(f"  [WATERMARK] Incremental load from: {last_watermark}")
    return df.filter(
        F.col(watermark_col) > F.lit(last_watermark).cast(TimestampType())
    )


In [0]:
def merge_scd1(df_new: DataFrame, target_table_full: str,
               primary_key: str):
    """
    SCD Type 1 — UPSERT.
    - On PK match AND data changed → update non-PK columns
    - On PK match but data unchanged → skip (no write)
    - No PK match → insert as new record

    Fix 1: Added change_condition to whenMatchedUpdate so unchanged
            records are not written — Delta skips them entirely.
    Fix 2: Returns Delta operation metrics so process_table can report
            accurate inserted/updated counts from Delta itself.
    """
    pk_cols = [c.strip() for c in primary_key.split(",")]
    now     = datetime.now(timezone.utc)

    # Add DWH audit columns
    df_new = (
        df_new
        .withColumn("dwh_created_at", F.lit(now).cast(TimestampType()))
        .withColumn("dwh_updated_at", F.lit(now).cast(TimestampType()))
    )

    merge_condition = " AND ".join(
        [f"target.{c} = source.{c}" for c in pk_cols]
    )

    # Columns to check for changes — exclude PKs, dwh_created_at (preserve), dwh_updated_at (always changes)
    exclude_from_compare = set(pk_cols + ["dwh_created_at", "dwh_updated_at"])
    compare_cols = [c for c in df_new.columns if c not in exclude_from_compare]

    # Only update when at least one business column actually changed
    # Handles: value changed, NULL→value, value→NULL
    change_condition = " OR ".join([
        f"(target.{c} <> source.{c}"
        f" OR (target.{c} IS NULL AND source.{c} IS NOT NULL)"
        f" OR (target.{c} IS NOT NULL AND source.{c} IS NULL))"
        for c in compare_cols
    ])

    # Update all non-PK columns except dwh_created_at (preserve original insert time)
    non_pk_cols = [c for c in df_new.columns
                   if c not in pk_cols and c != "dwh_created_at"]
    update_map  = {c: f"source.{c}" for c in non_pk_cols}
    insert_map  = {c: f"source.{c}" for c in df_new.columns}

    DeltaTable.forName(spark, target_table_full) \
        .alias("target") \
        .merge(df_new.alias("source"), merge_condition) \
        .whenMatchedUpdate(
            condition = change_condition,
            set       = update_map
        ) \
        .whenNotMatchedInsert(values=insert_map) \
        .execute()

    # Read exact counts from Delta operation history
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {target_table_full} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )

    print(f"  [SCD1] Merge complete → {target_table_full}")
    return metrics

In [0]:
def merge_scd2(df_new: DataFrame, target_table_full: str,
               primary_key: str):
    """
    SCD Type 2 — Full history preservation.

    Step 1 — Close changed current records:
      For records where PK matches AND is_current=true AND any business column changed
      → set is_current=False, effective_to=now

    Step 2 — Insert new and changed records:
      After Step 1, changed records have is_current=False so the merge
      condition (PK + is_current=true) no longer matches them.
      They are inserted as new rows with is_current=True.
      Truly new records (PK not in target) are also inserted.
      Unchanged records still match (is_current=True) → no action.

    Result:
      New record    : one row, is_current=True
      Changed record: old row closed (is_current=False, effective_to=now)
                      new row inserted (is_current=True, effective_from=now)
      Unchanged     : no change

    Returns:
      {"rows_closed": int, "rows_inserted": int}
      rows_closed   = from Step 1 (numTargetRowsUpdated)
      rows_inserted = from Step 2 (numTargetRowsInserted)
    """
    pk_cols = [c.strip() for c in primary_key.split(",")]
    now     = datetime.now(timezone.utc)

    df_new = (
        df_new
        .withColumn("effective_from", F.lit(now).cast(TimestampType()))
        .withColumn("effective_to",   F.lit(None).cast(TimestampType()))
        .withColumn("is_current",     F.lit(True))
        .withColumn("dwh_created_at", F.lit(now).cast(TimestampType()))
        .withColumn("dwh_updated_at", F.lit(now).cast(TimestampType()))
    )

    merge_condition = (
        " AND ".join([f"target.{c} = source.{c}" for c in pk_cols])
        + " AND target.is_current = true"
    )

    exclude_cols = set(pk_cols + SCD2_COLS)
    compare_cols = [c for c in df_new.columns if c not in exclude_cols]

    change_condition = " OR ".join([
        f"(target.{c} <> source.{c}"
        f" OR (target.{c} IS NULL AND source.{c} IS NOT NULL)"
        f" OR (target.{c} IS NOT NULL AND source.{c} IS NULL))"
        for c in compare_cols
    ])

    delta_target = DeltaTable.forName(spark, target_table_full)

    # Step 1: Close changed current records
    delta_target.alias("target") \
        .merge(df_new.alias("source"), merge_condition) \
        .whenMatchedUpdate(
            condition = change_condition,
            set = {
                "is_current"    : F.lit(False),
                "effective_to"  : F.lit(now).cast(TimestampType()),
                "dwh_updated_at": F.lit(now).cast(TimestampType())
            }
        ).execute()

    # Capture Step 1 metrics BEFORE Step 2 overwrites history
    step1_metrics = (
        spark.sql(f"DESCRIBE HISTORY {target_table_full} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )

    # Step 2: Insert new and changed records
    insert_map = {c: f"source.{c}" for c in df_new.columns}

    delta_target.alias("target") \
        .merge(df_new.alias("source"), merge_condition) \
        .whenNotMatchedInsert(values=insert_map) \
        .execute()

    # Capture Step 2 metrics
    step2_metrics = (
        spark.sql(f"DESCRIBE HISTORY {target_table_full} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )

    print(f"  [SCD2] Merge complete → {target_table_full}")

    return {
        "rows_closed":   int(step1_metrics.get("numTargetRowsUpdated",  0)),
        "rows_inserted": int(step2_metrics.get("numTargetRowsInserted", 0))
    }

In [0]:
def delete_stale_records(df_deduped: DataFrame, target_full: str,
                          primary_key: str, scd_type: str):
    """
    DELSERT delete step — only valid for load_type=FULL.

    Compares PKs in the current full source read (df_deduped, PRE-DQ)
    against PKs currently in silver, and removes/closes records that
    no longer exist in bronze (hard-deleted at source).

    Uses df_deduped (not df_clean) so DQ-rejected-but-still-present
    records are NOT wrongly flagged as deleted.

    SCD1: physically DELETE the row from silver.
    SCD2: close the current version (is_current=False, effective_to=now)
          — preserves history.

    Edge case handling:
    - NULL PK rows in source are excluded from comparison (left_anti
      misses NULLs anyway, and a NULL-PK source row is meaningless
      for delete detection).
    - Safety guard: if the source has 0 valid-PK rows, delete detection
      is SKIPPED entirely and a warning is logged. This prevents a
      bronze read error / empty table from wiping out all of silver.

    Returns: count of records deleted/closed.
    """
    pk_cols = [c.strip() for c in primary_key.split(",")]
    now     = datetime.now(timezone.utc)

    # ── Edge case 1: exclude NULL PK rows from source comparison set ──
    source_pks = df_deduped.select(*pk_cols).distinct()
    for c in pk_cols:
        source_pks = source_pks.filter(F.col(c).isNotNull())

    source_pk_count = source_pks.count()

    # ── Edge case 2: safety guard against empty/near-empty source ─────
    if source_pk_count == 0:
        print(f"  [DELSERT] SAFETY GUARD: source has 0 valid-PK records. "
              f"Skipping delete detection entirely to avoid wiping silver.")
        return 0

    delta_target = DeltaTable.forName(spark, target_full)

    if scd_type == "SCD2":
        target_current = (
            spark.table(target_full)
            .filter("is_current = true")
            .select(*pk_cols)
            .distinct()
        )
        for c in pk_cols:
            target_current = target_current.filter(F.col(c).isNotNull())

        stale_pks   = target_current.join(source_pks, on=pk_cols, how="left_anti")
        stale_count = stale_pks.count()

        if stale_count > 0:
            merge_condition = " AND ".join(
                [f"target.{c} = source.{c}" for c in pk_cols]
            ) + " AND target.is_current = true"

            delta_target.alias("target") \
                .merge(stale_pks.alias("source"), merge_condition) \
                .whenMatchedUpdate(set={
                    "is_current"    : F.lit(False),
                    "effective_to"  : F.lit(now).cast(TimestampType()),
                    "dwh_updated_at": F.lit(now).cast(TimestampType())
                }) \
                .execute()
            print(f"  [DELSERT] Closed {stale_count} stale current record(s) "
                  f"(SCD2 — history preserved, is_current set to False)")

    else:  # SCD1 — hard delete
        target_pks = spark.table(target_full).select(*pk_cols).distinct()
        for c in pk_cols:
            target_pks = target_pks.filter(F.col(c).isNotNull())

        stale_pks   = target_pks.join(source_pks, on=pk_cols, how="left_anti")
        stale_count = stale_pks.count()

        if stale_count > 0:
            merge_condition = " AND ".join(
                [f"target.{c} = source.{c}" for c in pk_cols]
            )
            delta_target.alias("target") \
                .merge(stale_pks.alias("source"), merge_condition) \
                .whenMatchedDelete() \
                .execute()
            print(f"  [DELSERT] Hard-deleted {stale_count} stale record(s) from silver")

    return stale_count

In [0]:
def process_table(config):
    """
    End-to-end processing for one bronze → silver table.
    Returns 'SUCCESS', 'FAILED', or 'SKIPPED' so main() can count correctly.

    Fixes applied:
    1. merge_scd1 returns Delta metrics — use numTargetRowsInserted/Updated
    2. NULL PK safety filter after DQ — catches NULLs that left_anti missed
    3. Returns status string so main() counts correctly
    4. Watermark read from metadata_silver_config (get_last_watermark)
    5. load_type and write_mode are independent
    6. Incremental filter on SOURCE col, deduplicate on MAPPED col
    7. MAX watermark captured before DQ and written back on success
    8. HASH PK: generate_hash_key() runs on bronze BEFORE mapping,
       and the generated column is passed through apply_column_mapping()
       via passthrough_cols — otherwise it gets dropped by select().
    """
    source_table     = config["source_table"]
    target_table     = config["target_table"]
    source_schema    = config["source_schema"]
    target_schema    = config["target_schema"]
    load_type        = config["load_type"]
    write_mode       = config["write_mode"]
    scd_type         = config["scd_type"]
    primary_key      = config["primary_key"]
    pk_strategy      = config["pk_strategy"] or "NATURAL"
    hash_columns     = config["hash_columns"]
    hash_column_name = config["hash_column_name"]
    watermark_col    = config["watermark_column"]

    source_full     = f"{CATALOG}.{source_schema}.{source_table}"
    target_full     = f"{CATALOG}.{target_schema}.{target_table}"
    start_time      = datetime.now(timezone.utc)
    was_forced_full = False

    records_read = records_inserted = records_updated = 0
    records_rejected = dq_pass_count = dq_fail_count = 0
    records_deleted = 0

    print(f"\n{'='*60}")
    print(f"[START] {source_full} → {target_full}")
    print(f"  load_type={load_type} | write_mode={write_mode} | "
          f"scd_type={scd_type} | pk_strategy={pk_strategy} | pk={primary_key}")

    try:
        # ── Step 1: Load column mapping ───────────────────────────────────
        mappings = load_column_mapping(source_table, target_table)
        if not mappings:
            raise ValueError(f"No active column mappings found for {source_table} → {target_table}")

        col_name_map         = {m["source_column"]: m["target_column"] for m in mappings}
        watermark_col_mapped = col_name_map.get(watermark_col, watermark_col)

        # ── Step 2: Get watermark (INCREMENTAL only) ──────────────────────
        last_watermark = None
        if load_type == "INCREMENTAL" and watermark_col:
            last_watermark = get_last_watermark(source_table)
            if last_watermark is None:
                was_forced_full = True
                print(f"  [WATERMARK] First run — no prior watermark. Treating as FULL LOAD.")

        # ── Step 3: Read bronze ───────────────────────────────────────────
        df_bronze    = spark.table(source_full)
        records_read = df_bronze.count()
        print(f"  [READ] Bronze records: {records_read}")

        if records_read == 0:
            print(f"  [SKIP] No records in source. Skipping.")
            write_audit_log(
                BATCH_ID, PIPELINE_ID, source_table, target_table,
                SOURCE_SCHEMA, TARGET_SCHEMA, start_time, datetime.now(timezone.utc),
                0, 0, 0, 0, 0, 0,
                load_type, write_mode, scd_type, pk_strategy,
                was_forced_full, "SKIPPED", "Empty source table"
            )
            return "SKIPPED"

        # ── Step 4: Apply incremental filter on SOURCE col name ───────────
        if load_type == "INCREMENTAL" and watermark_col:
            df_bronze         = apply_incremental_filter(df_bronze, watermark_col, last_watermark)
            incremental_count = df_bronze.count()
            print(f"  [FILTER] After watermark filter: {incremental_count} records")
            if incremental_count == 0 and not was_forced_full:
                print(f"  [SKIP] No new/updated records since last run.")
                write_audit_log(
                    BATCH_ID, PIPELINE_ID, source_table, target_table,
                    SOURCE_SCHEMA, TARGET_SCHEMA, start_time, datetime.now(timezone.utc),
                    records_read, 0, 0, 0, 0, 0,
                    load_type, write_mode, scd_type, pk_strategy,
                    was_forced_full, "SKIPPED", "No new records since last watermark"
                )
                return "SKIPPED"
        else:
            print(f"  [FILTER] FULL load — processing all {records_read} records")

        # ── Step 5: Generate HASH surrogate key BEFORE mapping ────────────
        # Must run on bronze df since hash_columns are SOURCE column names.
        # The generated column is preserved through mapping via passthrough_cols.
        passthrough = None
        if pk_strategy == "HASH" and hash_columns and hash_column_name:
            df_bronze   = generate_hash_key(df_bronze, hash_columns, hash_column_name)
            passthrough = [hash_column_name]
            print(f"  [HASH] Generated surrogate key '{hash_column_name}' "
                  f"from columns: {hash_columns}")
        elif pk_strategy == "HASH":
            raise ValueError(
                f"pk_strategy=HASH but hash_columns or hash_column_name is missing. "
                f"hash_columns='{hash_columns}', hash_column_name='{hash_column_name}'. "
                f"Check metadata_silver_config for {source_table}."
            )

        # ── Step 6: Apply column mapping + transformations ────────────────
        df_mapped = apply_column_mapping(df_bronze, mappings, passthrough_cols=passthrough)
        print(f"  [MAP] Column mapping applied. Output columns: {len(df_mapped.columns)}")

        # ── Step 7: Deduplicate using MAPPED watermark col name ───────────
        df_deduped  = deduplicate(df_mapped, primary_key, watermark_col_mapped)
        dup_removed = df_mapped.count() - df_deduped.count()
        if dup_removed > 0:
            print(f"  [DEDUP] Removed {dup_removed} duplicate records")

        # ── Step 8: Capture MAX watermark BEFORE DQ ───────────────────────
        max_watermark = None
        if watermark_col_mapped and watermark_col_mapped in df_deduped.columns:
            max_wm_row = df_deduped.agg(F.max(watermark_col_mapped).alias("max_wm")).collect()
            if max_wm_row and max_wm_row[0]["max_wm"] is not None:
                max_watermark = max_wm_row[0]["max_wm"]

        # ── Step 9: DQ checks ─────────────────────────────────────────────
        dq_rules = load_dq_rules(target_table)
        print(f"  [DQ] Running {len(dq_rules)} DQ rule(s)...")

        pk_col_for_log = [c.strip() for c in primary_key.split(",")][0]

        if dq_rules:
            df_clean, dq_pass_count, dq_fail_count = run_dq_checks(
                df         = df_deduped,
                rules      = dq_rules,
                pk_col     = pk_col_for_log,
                table_name = target_table,
                batch_id   = BATCH_ID
            )
        else:
            df_clean = df_deduped
            print(f"  [DQ] No DQ rules configured — skipping DQ")

        # ── Step 10: NULL PK safety filter ────────────────────────────────
        # left_anti join in run_dq_checks cannot exclude NULL PKs (NULL != NULL)
        # Delta NOT NULL constraint will reject them — filter explicitly here
        null_pk_count = df_clean.filter(F.col(pk_col_for_log).isNull()).count()
        if null_pk_count > 0:
            print(f"  [PK-FILTER] Removed {null_pk_count} record(s) with NULL PK")
            df_clean = df_clean.filter(F.col(pk_col_for_log).isNotNull())

        records_rejected = df_deduped.count() - df_clean.count()
        print(f"  [DQ] Pass={dq_pass_count} | Fail={dq_fail_count} | Rejected={records_rejected}")

        # ── Step 11: Write to Silver ───────────────────────────────────────
        clean_count = df_clean.count()

        if write_mode == "UPSERT":
            if scd_type == "SCD1":
                merge_metrics    = merge_scd1(df_clean, target_full, primary_key)
                records_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
                records_updated  = int(merge_metrics.get("numTargetRowsUpdated",  0))

            elif scd_type == "SCD2":
                merge_metrics    = merge_scd2(df_clean, target_full, primary_key)
                records_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
                records_updated  = int(merge_metrics.get("numTargetRowsUpdated",  0))  

            else:
                raise ValueError(
                    f"write_mode=UPSERT requires scd_type SCD1 or SCD2. Got '{scd_type}'"
                )

        elif write_mode == "OVERWRITE":
            now = datetime.now(timezone.utc)
            df_clean = (
                df_clean
                .withColumn("dwh_created_at", F.lit(now).cast(TimestampType()))
                .withColumn("dwh_updated_at", F.lit(now).cast(TimestampType()))
            )
            df_clean.write.format("delta").mode("overwrite") \
                .option("overwriteSchema", "false") \
                .saveAsTable(target_full)
            records_inserted = clean_count

        elif write_mode == "DELSERT":
            if load_type != "FULL":
                raise ValueError(
                    f"write_mode=DELSERT requires load_type=FULL "
                    f"(needs complete source set to detect deletions). "
                    f"Got load_type='{load_type}'."
                )

            if scd_type == "SCD1":
                merge_metrics    = merge_scd1(df_clean, target_full, primary_key)
                records_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
                records_updated  = int(merge_metrics.get("numTargetRowsUpdated",  0))

            elif scd_type == "SCD2":
                scd2_metrics     = merge_scd2(df_clean, target_full, primary_key)
                records_inserted = scd2_metrics["rows_inserted"]
                records_updated  = scd2_metrics["rows_closed"]

            else:
                raise ValueError(
                    f"write_mode=DELSERT requires scd_type SCD1 or SCD2. Got '{scd_type}'"
                )

            records_deleted = delete_stale_records(df_deduped, target_full, primary_key, scd_type)
                    
        else:
            raise ValueError(f"Unknown write_mode='{write_mode}'. Expected UPSERT, DELSERT, or OVERWRITE.")

        end_time = datetime.now(timezone.utc)

        # ── Step 12: Update watermark ──────────────────────────────────────
        if max_watermark is not None:
            update_watermark(source_table, max_watermark, end_time)

        # ── Step 13: Audit log ────────────────────────────────────────────
        write_audit_log(
            BATCH_ID, PIPELINE_ID, source_table, target_table,
            SOURCE_SCHEMA, TARGET_SCHEMA, start_time, end_time,
            records_read, records_inserted, records_updated,
            records_rejected, records_deleted, dq_pass_count, dq_fail_count,
            load_type, write_mode, scd_type, pk_strategy,
            was_forced_full, "SUCCESS"
        )
        print(f"[DONE] {source_table} → SUCCESS")
        return "SUCCESS"

    except Exception as e:
        end_time      = datetime.now(timezone.utc)
        error_message = traceback.format_exc()
        print(f"[FAILED] {source_table} → {str(e)}")
        write_audit_log(
            BATCH_ID, PIPELINE_ID, source_table, target_table,
            SOURCE_SCHEMA, TARGET_SCHEMA, start_time, end_time,
            records_read, 0, 0, 0, 0, 0,0,
            load_type, write_mode, scd_type, pk_strategy,
            was_forced_full, "FAILED", traceback.format_exc()
        )
        return "FAILED"

In [0]:
def main():
    print("=" * 60)
    print(f"  WCL Bronze → Silver Pipeline")
    print(f"  Batch ID  : {BATCH_ID}")
    print(f"  Started   : {RUN_TIMESTAMP}")
    print(f"  Run Mode  : {RUN_MODE}")
    print(f"  Filter    : {TABLE_FILTER if TABLE_FILTER else '(all tables)'}")
    print("=" * 60)

    configs = load_table_configs()

    if not configs:
        print("[WARN] No active table configurations found. Check metadata_silver_config.")
        return

    success_count = 0
    failed_count  = 0
    skipped_count = 0

    for config in configs:
        # process_table returns 'SUCCESS', 'FAILED', or 'SKIPPED'
        # It never raises — all exceptions are caught internally
        status = process_table(config)
        if status == "SUCCESS":
            success_count += 1
        elif status == "SKIPPED":
            skipped_count += 1
        else:
            failed_count += 1

    print("\n" + "=" * 60)
    print(f"  Pipeline Complete")
    print(f"  Total Tables : {len(configs)}")
    print(f"  Success      : {success_count}")
    print(f"  Skipped      : {skipped_count}")
    print(f"  Failed       : {failed_count}")
    print(f"  Batch ID     : {BATCH_ID}")
    print("=" * 60)

In [0]:
main()